In [ ]:
from pathlib import Path
from datetime import datetime, date, timedelta
import xarray as xr

from wekeo_combined_chain import config

from wekeo_combined_chain.utils import select_area

In [ ]:
# """
# This Cell is made to build a multi-day dataset from the gridded netcdf datasets
# TODO: - Replace by S3 zarr access
# """

# def get_combined_ds_path(day: date) -> Path:
#     base = config.gridded_combined_downloaded_dir
#     return base / day.strftime("%Y_%m_%d/COMBINED_%Y-%m-%d_v1.0.nc")\

# def remove_vars(ds: xr.Dataset, vars_to_not_remove: list[str]) -> xr.Dataset:
#     # general rules of vars to remove:
    
#     remove = []
#     vars = ds.data_vars.keys()
    
#     patterns = ["__night", "_std", "_max", "_min", "_count"]
#     for var in vars:
#         if any([pattern in var for pattern in patterns]) and var not in vars_to_not_remove:
#             remove.append(var)
    
#     ds = ds.drop_vars(remove)
#     return ds
    
# def get_combined_ds_range(
#     area: list[float] = [ 90., -90.,  180., -180.],
#     start_day: date = date(2025, 1, 1), 
#     end_day: date = date(2025, 12, 31),
#     ) -> xr.Dataset:

#     # iterate dates
#     days = [start_day + i*timedelta(days=1) for i in range((end_day - start_day).days + 1)]
#     paths = [get_combined_ds_path(day) for day in days]
    
#     valid_pairs = []
#     for day, path in zip(days, paths):
#         if not path.exists():
#             print(f"/!\\ Warning: File {path} does not exist.")
#         else:
#             valid_pairs.append((day, path))

    
#     # Not great, but since the datasets does not have a time dimension this is needed.
#     # Since we need to load the dataset, lets also take the time to apply the area selection.
#     datasets = []
#     for day, path in valid_pairs:
#         ds = xr.open_dataset(path)
#         ds = remove_vars(ds, vars_to_not_remove=[])
#         ds = select_area(ds, area)
#         ds = ds.expand_dims(time=[datetime(day.year, day.month, day.day)])
#         datasets.append(ds)

    # return xr.concat(datasets, dim="time")


In [ ]:
"""
S3 Zarr access variant
"""
from datetime import datetime, date
import s3fs
import xarray as xr

from wekeo_combined_chain.utils import select_area


def _make_fs(endpoint_url: str) -> s3fs.S3FileSystem:
    return s3fs.S3FileSystem(
        anon=False,
        client_kwargs={"endpoint_url": endpoint_url},
        config_kwargs={"max_pool_connections": 20},
        s3_additional_kwargs={"retries": {"max_attempts": 5, "mode": "adaptive"}},
    )


def _open_combined_zarr() -> xr.Dataset:
    """
    Open the yearly concatenated global COMBINED Zarr store on S3.
    Time is unordered, so callers must sort/selection by time explicitly.
    """
    BUCKET = "S5P_PCA_V0.1"
    ZARR_PATH = f"s3://{BUCKET}/COMBINED/v1.0/COMBINED_dataset_v0.1.zarr"
    fs = _make_fs("https://s3.waw4-1.cloudferro.com")
    store = fs.get_mapper(ZARR_PATH)
    # consolidated=True for fast metadata read; chunks stay lazy on S3
    return xr.open_zarr(store, consolidated=True)


def remove_vars(ds: xr.Dataset, vars_to_not_remove: list[str]) -> xr.Dataset:
    # general rules of vars to remove:
    remove = []
    vars = ds.data_vars.keys()

    patterns = ["__night", "_std", "_max", "_min", "_count"]
    for var in vars:
        if any([pattern in var for pattern in patterns]) and var not in vars_to_not_remove:
            remove.append(var)

    ds = ds.drop_vars(remove)
    return ds


def get_combined_ds_range(
    area: list[float] = [90., -90., 180., -180.],
    start_day: date = date(2025, 1, 1),
    end_day: date = date(2025, 12, 31),
) -> xr.Dataset:
    """
    Load a date range from the S3 Zarr store and select the requested area.

    The Zarr store is a yearly concatenated global file with unordered time,
    so we sort by time before selecting the [start_day, end_day] slice.
    Area selection is applied lazily on the resulting slice.
    """
    ds = _open_combined_zarr()

    # The store time is unordered: sort once so .sel(time=slice(...)) is reliable.
    ds = ds.sortby("time")

    # Build a [start, end] inclusive slice at day granularity.
    start = datetime(start_day.year, start_day.month, start_day.day)
    end = datetime(end_day.year, end_day.month, end_day.day)
    ds = ds.sel(time=slice(start, end))

    ds = remove_vars(ds, vars_to_not_remove=[])
    ds = select_area(ds, area)

    return ds


In [ ]:
areas = {
    "Global":          [ 90., -90.,  180., -180.],
    "North_America":   [ 90.,   9.,  -20., -169.],
    "South_America":   [  9., -60.,  -20.,  120.],
    "Europe":          [ 90.,  36.,   31.,  -20.],
    "Africa":          [ 36., -60.,   60.,  -20.],
    "Russia":          [ 90.,  36., -169.,   31.],
    "Asia":            [ 36., -10., -169.,   60.],
    "Australia":       [-10., -60., -120.,   60.],
    # "Central_Pacific": [  9., -10., -120., -169.],
    # "Antarctic":       [-60., -90.,  180., -180.],
}

area_name = "Global"
area = areas[area_name]

## User defined area override example:
# area_name = "France"
# area = [53., 41., 9., -5.]

start = date(2025, 7, 25)
end = date(2025, 8, 5)

ds = get_combined_ds_range(area, start, end)

# Plot timeseries of plume detections

In [ ]:
from wekeo_combined_chain.timeseries.plot import plot_plume_timeseries

fig, ax = plot_plume_timeseries(ds)

# Plot timeseries of pixel counts for plumes

In [ ]:
from wekeo_combined_chain.timeseries.plot import plot_plume_pixels_timeseries

# Plot timeseries of pixel counts for normal plumes
fig, ax = plot_plume_pixels_timeseries(ds, plume_size="normal")

# Plot timeseries of pixel counts for tiny plumes
fig, ax = plot_plume_pixels_timeseries(ds, plume_size="tiny")

# Plot timeseries of daily detected pixels

In [ ]:
from wekeo_combined_chain.timeseries.plot import plot_detected_pixels_timeseries

# Plot timeseries of daily detected pixels (s5p_pca__mean_score_CO not NaN)
fig, ax = plot_detected_pixels_timeseries(ds)

# Plot timeseries of daily FRP pixels

In [ ]:
from wekeo_combined_chain.timeseries.plot import plot_frp_pixels_timeseries

# Plot timeseries of daily FRP pixels (day/night MWIR not NaN)
fig, ax = plot_frp_pixels_timeseries(ds)

# Occurrence Maps

In [ ]:
"""
Coarsen dataset to reduce spatial resolution by block-averaging.
"""

FACTOR = 12 # 12-16 good on a global scale, for regionnal smaller factor advised, around 4-8
dsc = ds.coarsen(latitude=FACTOR, longitude=FACTOR, boundary="trim").mean()

# Plume Occurence Map

In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import plot_plume_occurrence_map

fig, ax = plot_plume_occurrence_map(dsc)


# Detection Occurence Map

In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import plot_detection_occurrence_map

fig, ax = plot_detection_occurrence_map(dsc)


# Active Fire Occurence

In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import plot_fire_occurrence_map

# Note: uses frp_slstr__day_FRP_MWIR_mean (and/or night) to detect active fire days
fig, ax = plot_fire_occurrence_map(dsc)


# Animated Daily Maps

Browse day-by-day through the spatial maps using an interactive slider.
Each function also accepts `save_gif_path` to export the animation as a GIF file.


In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import animate_plume_map

# Interactive slider — scrub through days
animate_plume_map(dsc)


In [ ]:
# Export as GIF (uncomment to run)
# animate_plume_map(ds, save_gif_path="plume_animation.gif", fps=2, dpi=100)


In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import animate_detection_map

animate_detection_map(dsc)
